#Take-home exercise 04
Chunlin An | ca2965

In [0]:
# import libraries
from pyspark.sql.functions import col, year, current_date, datediff, round, avg, min, max, rank, upper, substring, when, isnull, concat, udf, create_map, lit, floor, sum, countDistinct, count
from pyspark.sql.window import Window
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from pyspark.ml.evaluation import RegressionEvaluator
%pip install statsmodels

mlflow.set_tracking_uri("databricks")

In [0]:
# read in data
df_laptimes = spark.read.csv('/Volumes/gr5069/raw/f1_data/lap_times.csv', header = True)

df_drivers = spark.read.csv('/Volumes/gr5069/raw/f1_data/drivers.csv', header = True)

df_pitstops = spark.read.csv('/Volumes/gr5069/raw/f1_data/pit_stops.csv', header = True)

df_results = spark.read.csv('/Volumes/gr5069/raw/f1_data/results.csv', header = True)

## Data preparation
I will use `df_pitstop` to calcualte average pit stop duration per driver, join with `df_results` to use for subsequent modeling and predictions

In [0]:
display(df_results)

In [0]:
display(df_pitstops)

In [0]:
# cast duration time (milliseconds) as double, then convert to seconds for clarity
df_clean = df_pitstops.withColumn("duration_seconds", round(col("milliseconds").cast("double")/1000, 2))

# group by driver id, race id and calculate average pit stop time & number of stops

df_avg_pit = df_clean.groupBy("driverId", "raceId").agg(
    avg("duration_seconds").alias("avg_pitstop_time"),
    count("stop").alias("num_stops"))

df_avg_pit.show(5)

In [0]:
# join with results table on driverId

df_results_pit = df_results.drop(*['number', 'positionOrder', 'positionText','constructorId', "resultId", 'statusId', 'time', 'milliseconds' ]).join(df_avg_pit.select('driverId', 'raceId', 'avg_pitstop_time', 'num_stops'), on \
    = ['driverId', 'raceId'], how = 'left').dropna()

df_results_pit.show(5)

In [0]:
display(df_results_pit)

In [0]:
df = df_results_pit.toPandas()

# clean position column - some values may be \N
df = df[pd.to_numeric(df["position"], errors="coerce").notna()]
df["position"] = df["position"].astype(int)
df["points"] = df["points"].astype(float)
df["grid"] = df["grid"].astype(int)

In [0]:
df["top10"] = (df["position"].astype(int) <= 10).astype(int)
df.head()

## Machine Learning

In [0]:

mlflow.set_experiment("/Workspace/Users/ca2965@columbia.edu/HW/take-home-exercise-4-acl13820-cmd/F1_RF")

### Model 1: predict finishing in top 10 (position <= 10) using `avg_pitstop_time` and `num_stops` with Logistic Regression

In [0]:
# MODEL 1 
X1 = df[["avg_pitstop_time", "num_stops"]]
y1 = df["top10"]
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

In [0]:
# hyperparamters

param_grid_lr = [
    {"fit_intercept": True, "C": 0.1, "solver": "liblinear"},
    {"fit_intercept": True, "C": 1.0, "solver": "liblinear"},
    {"fit_intercept": True, "C": 10.0, "solver": "liblinear"},
]



In [0]:
# define experiments 

def log_lr(experimentID, run_name, params, X_train, X_test, y_train, y_test):
    from pyspark.sql.functions import lit
    from sklearn.metrics import precision_score, recall_score
    import matplotlib.pyplot as plt
    import mlflow
    import mlflow.sklearn
    import seaborn as sns
    import pandas as pd
    import tempfile

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score, precision_score, recall_score,
        f1_score, roc_auc_score, ConfusionMatrixDisplay
    )

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
        
        # train Logistic Regression
        lr = LogisticRegression(**params, random_state=42, class_weight="balanced", max_iter=1000)
        lr.fit(X_train, y_train)
    
        predictions = lr.predict(X_test)
        proba = lr.predict_proba(X_test)

        # store predictions as Pandas df 
        testPDF = X_test.copy()
        testPDF["label"] = y_test
        testPDF["prediction"] = predictions
        testPDF["predict_prob"] = proba[:,1]
        testPDF["run_id"] = run.info.run_id
        testPDF["experiment_id"] = run.info.experiment_id

        # write predictions as table
        predDF = spark.createDataFrame(testPDF)
        predDF = predDF.withColumn("model", lit("logistic_regression"))
        predDF.write.mode("append").saveAsTable("gr5069.ca2965.f1_lr_predictions")

        # log model
        mlflow.sklearn.log_model(lr, "logistic-regression-model")

        # log params
        for param, value in params.items():
            mlflow.log_param(param, value)

        # metrics
        acc  = accuracy_score(y_test, predictions)
        prec = precision_score(y_test, predictions, average = 'binary')
        rec  = recall_score(y_test, predictions, average = 'binary')
        f1   = f1_score(y_test, predictions,average = 'binary')
        auc = roc_auc_score(y_test, proba[:, 1])

        print(f"accuracy: {acc}")
        print(f"precision: {prec}")
        print(f"recall: {rec}")
        print(f"f1: {f1}")
        print(f"auc: {auc}")

        mlflow.log_metric("accuracy",  acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall",    rec)
        mlflow.log_metric("f1",        f1)
        mlflow.log_metric("auc",       auc)

        # artifact 1: coefficients instead of feature importance
        importance = pd.DataFrame({
            "Feature": X_train.columns,
            "Coefficient": lr.coef_[0]
        })

        temp = tempfile.NamedTemporaryFile(prefix="coefficients-", suffix=".csv")
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "coefficients.csv")
        finally:
            temp.close()

        # artifact 2: confusion matrix
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=ax)
        ax.set_title(f"Confusion Matrix (LR)")

        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png")
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "confusion-matrix.png")
        finally:
            temp.close()

        display(fig)

        return run.info.run_id

In [0]:
# set experiment 
experiment1_path = "/Users/ca2965@columbia.edu/HW/take-home-exercise-4-acl13820-cmd/F1_LR"

mlflow.set_experiment(experiment1_path)

experimentID = mlflow.get_experiment_by_name(experiment1_path).experiment_id

for i, params in enumerate(param_grid_lr):
    run_name = f"run_{i+1}_C{params['C']}_intercept{params['fit_intercept']}"
    
    log_lr(
        experimentID,
        run_name,
        params,
        X1_train,
        X1_test,
        y1_train,
        y1_test
    )


### Model 2: predict finishing top 10 from `grid`, `avg_pitstop_time`, and `num_stops` with a Random Forest Classifier.

In [0]:
# MODEL 2
X2 = df[["grid", "avg_pitstop_time"]]
y2 = df["top10"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)


In [0]:
# hyperparameters
param_grid_rf = [
    {"n_estimators": 50,  "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
]


In [0]:
def log_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test):
    import pandas as pd
    from pyspark.sql.functions import lit
    import matplotlib.pyplot as plt
    import mlflow.sklearn
    import seaborn as sns
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, ConfusionMatrixDisplay
    import tempfile

    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:
        # train
        rf = RandomForestClassifier(**params, random_state=42, class_weight="balanced")
        rf.fit(X_train, y_train)
        
        predictions = rf.predict(X_test)
        proba = rf.predict_proba(X_test)[:, 1]

        # store predictions as Pandas df 
        testPDF = X_test.copy()
        testPDF["label"] = y_test
        testPDF["prediction"] = predictions
        testPDF["predict_prob"] = proba
        testPDF["run_id"] = run.info.run_id
        testPDF["experiment_id"] = run.info.experiment_id

        # write predictions as 
        predDF = spark.createDataFrame(testPDF)
        predDF = predDF.withColumn("model", lit("random_forest_classification"))
        predDF.write.mode("append").saveAsTable("gr5069.ca2965.f1_rf_predictions")

        # log model
        mlflow.sklearn.log_model(rf, "random-forest-model")

        # log params
        [mlflow.log_param(param, value) for param, value in params.items()]

        # metrics
        acc  = accuracy_score(y_test, predictions)
        prec = precision_score(y_test, predictions)
        rec  = recall_score(y_test, predictions)
        f1   = f1_score(y_test, predictions)
        auc  = roc_auc_score(y_test, proba)

        print(f"  accuracy: {acc}")
        print(f"  precision: {prec}")
        print(f"  recall: {rec}")
        print(f"  f1: {f1}")
        print(f"  auc: {auc}")

        mlflow.log_metric("accuracy",  acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall",    rec)
        mlflow.log_metric("f1",        f1)
        mlflow.log_metric("auc",       auc)

        # artifact 1: feature importance csv
        importance = pd.DataFrame({
            "Feature": X_train.columns,
            "Importance": rf.feature_importances_})
        
        temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")
        temp_name = temp.name
        try:
            importance.to_csv(temp_name, index=False)
            mlflow.log_artifact(temp_name, "feature-importance.csv")
        finally:
            temp.close()

        # artifact 2: confusion matrix plot
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_test, predictions, ax=ax)
        ax.set_title(f"Confusion Matrix (trees={params['n_estimators']}, depth={params['max_depth']})")

        temp = tempfile.NamedTemporaryFile(prefix="confusion-matrix-", suffix=".png")
        temp_name = temp.name
        try:
            fig.savefig(temp_name)
            mlflow.log_artifact(temp_name, "confusion-matrix.png")
        finally:
            temp.close()

        display(fig)
        return run.info.run_id


In [0]:
# set experiment 
experiment2_path = "/Users/ca2965@columbia.edu/HW/take-home-exercise-4-acl13820-cmd/F1_RF"

mlflow.set_experiment(experiment2_path)

experiment2ID = mlflow.get_experiment_by_name(experiment2_path).experiment_id

for i, params in enumerate(param_grid_rf):
    run_name = f"run_{i+1}_trees{params['n_estimators']}_depth{params['max_depth']}"
    log_rf(experiment2ID, run_name, params, X2_train, X2_test, y2_train, y2_test)
